# 05 · Robustness checks

Each cause proposed in the error analysis is tested by intervention, and the baseline is retrained with different feature sets. Model outputs are cached by `scripts/run_robustness.py`; rerunning this notebook recomputes the statistics from them.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, display, Markdown
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 90)

In [2]:
import subprocess
out = subprocess.run([sys.executable, str(ROOT/'scripts/run_robustness.py')], capture_output=True, text=True, cwd=ROOT)
print(out.returncode)

0


In [3]:
r = json.loads((ROOT/'reports/metrics/robustness.json').read_text())
pd.DataFrame({k: {m: v for m, v in d.items() if not isinstance(v, dict)} for k, d in r['R1_emoji_ablation'].items() if isinstance(d, dict)})

,english_only,xlmt
macro_f1_with_emoji,0.3619,0.4337
macro_f1_emoji_removed,0.1612,0.2721
pred_neutral_with_emoji,0.6959,0.4866
pred_neutral_emoji_removed,0.9246,0.6326
nonneutral_predictions_with_emoji,125.0000,211.0000
of_which_become_neutral,98.0000,96.0000
unchanged_on_emoji_free_tweets,538.0000,538.0000
n_emoji_free,538.0000,538.0000


## R2: rewriting `3` as ɛ

In [4]:
pd.DataFrame(r['R2_spelling_fix']['examples'])

,original,normalised
0,ah ka s3 prison mpo er,ah ka sɛ prison mpo er
1,slim things de3 3noaa ne wei ooo🤩🤩🤩😘😂,slim things deɛ ɛnoaa ne wei ooo🤩🤩🤩😘😂
2,ceremonial vp nso y3di bor bra,ceremonial vp nso yɛdi bor bra
3,eeeeiii 3ne3 mo mmoa nnipa na prostate de3 3ya,eeeeiii ɛneɛ mo mmoa nnipa na prostate deɛ ɛya
4,sei a na mep3 cmo,sei a na mepɛ cmo


In [5]:
pd.DataFrame({k: {m: v for m, v in d.items() if not isinstance(v, dict)} for k, d in r['R2_spelling_fix'].items() if isinstance(d, dict)})

,nllb_twi,xlmt
macro_f1_before,0.3417,0.3224
macro_f1_after,0.3491,0.3618
pred_neutral_before,0.5323,0.6537
pred_neutral_after,0.4729,0.5401
translations_with_3_spellings_before,188.0000,NaN
translations_with_3_spellings_after,2.0000,NaN


In [6]:
{k: d['mcnemar_before_vs_after'] for k, d in r['R2_spelling_fix'].items() if isinstance(d, dict)}

{'nllb_twi': {'only_a_correct': 46,
  'only_b_correct': 50,
  'p_value': 0.7596492883236297},
 'xlmt': {'only_a_correct': 18,
  'only_b_correct': 35,
  'p_value': 0.027008317653722358}}

## R3: twi_Latn vs aka_Latn

In [7]:
r['R3_twi_vs_aka']

{'macro_f1_twi': 0.3728,
 'macro_f1_aka': 0.3665,
 'mcnemar': {'only_a_correct': 84,
  'only_b_correct': 78,
  'p_value': 0.6945673674329714}}

## R4: baseline feature ablation

In [8]:
pd.DataFrame(r['R4_feature_ablation']).T

,chosen_C,dev_macro_f1,test_macro_f1,test_macro_f1_ci95
char_only,8.0,0.6146,0.5777,"[0.548, 0.6086]"
word_only,4.0,0.5662,0.5689,"[0.5373, 0.6018]"
char_plus_word (reported baseline),4.0,0.5995,0.5925,"[0.5591, 0.6251]"
char_plus_word_keeping_emoji,4.0,0.5993,0.5848,"[0.553, 0.6166]"
